In [1]:
# a. Set up environment
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model

# b. Load and preprocess data
(x_train,y_train),(x_test,y_test) = tf.keras.datasets.fashion_mnist.load_data()

x_train = np.repeat(x_train[...,None],3,axis=-1)
x_test = np.repeat(x_test[...,None],3,axis=-1)

x_train = tf.image.resize(x_train,(224,224)).numpy()
x_test = tf.image.resize(x_test,(224,224)).numpy()

x_train = preprocess_input(x_train)
x_test = preprocess_input(x_test)

# c. Load pre-trained VGG16
base = VGG16(weights="imagenet",include_top=False,
             input_shape=(224,224,3))

# d. Freeze base layers and add custom layers
base.trainable = False

x = GlobalAveragePooling2D()(base.output)
x = Dense(128,activation="relu")(x)
x = Dropout(0.5)(x)
output = Dense(10,activation="softmax")(x)

model = Model(base.input,output)

# e. Train model
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

history = model.fit(x_train,y_train,epochs=5,
                    validation_split=0.2)

# f, g. Compute loss and evaluate model
loss,accuracy = model.evaluate(x_test,y_test)
print("Test Loss:",loss)
print("Test Accuracy:",accuracy)

# h. Visualize predictions
predictions = model.predict(x_test[:5])

for i in range(5):
    plt.imshow(x_test[i][:,:,0],cmap="gray")
    plt.title("Predicted: "+str(np.argmax(predictions[i])))
    plt.axis("off")
    plt.show()

# i. Fine-tune model
base.trainable = True

for layer in base.layers[:-4]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

model.fit(x_train,y_train,epochs=5,validation_split=0.2)

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
 3612672/26421880 ━━━━━━━━━━━━━━━━━━━━ 11s 0us/step

KeyboardInterrupt: 